# Практическая работа №1. Часть 5Р
## Рекуррентные слои: РНС, LSTM, GRU

**Цель:** написать рекуррентную ячейку на «голых» тензорах и сверить ее с `nn.RNN`, увидеть на числах, почему простая рекуррентная сеть «забывает» начало последовательности, сравнить ее с LSTM на задаче на память и применить рекуррентные сети к временному ряду и к тексту.

> **Правило части.** В заданиях Р1–Р3 внутри своих функций – только операции тензоров (`@`, `+`, `torch.tanh`, `torch.stack`, срезы); `nn.RNN` используется лишь для сверки в Р2. Начиная с Р4 – готовые модули `nn.RNN`, `nn.LSTM`, `nn.Linear` и оптимизаторы.

**Задача:** сквозной пример со слайдов – одномерная рекуррентная сеть $h_t = \tanh(0{,}5\,h_{t-1} + 1 \cdot x_t)$ со входом $(1, 0, 0, 0, 0)$: одиночный импульс на первом шаге, дальше тишина. Затем три настоящие задачи: запомнить первый символ, предсказать следующее значение зашумленной синусоиды и сгенерировать текст по одной букве.

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом – подсказка написана в комментарии `# <--`. Напечатанное значение должно совпасть с тем, что стоит после слова «ожидается»; для обученных сетей указан порог. Ячейки обучения выполняются до минуты. Сначала прочитайте теорию ниже.

## Как сеть читает последовательности

**1. Почему полносвязная сеть не подходит.** Текст, речь, показания датчиков беспилотника, курс валют – это **последовательности** $x_1, x_2, \dots, x_T$, где важен порядок и длина бывает разной. Полносвязной сети из части 1С нужен вход фиксированной длины, и признак «слово “не” стоит перед глаголом» ей пришлось бы учить отдельно для каждой позиции. Нужна сеть, которая читает последовательность **шаг за шагом** и помнит, что прочитала.

**2. Ячейка рекуррентной сети (РНС).** На каждом шаге $t$ сеть получает очередной вход $x_t$ и свое прошлое **скрытое состояние** $h_{t-1}$ – вектор-«память» – и считает новое состояние:

$$h_t = \tanh\big(W h_{t-1} + U x_t + b\big).$$

Здесь $W$ – матрица $H \times H$ (как память переходит в память), $U$ – матрица $H \times D$ (как вход записывается в память), $b$ – смещение, $H$ – размер памяти, $D$ – размер входа. Начальное состояние $h_0$ обычно нулевое. Функция $\tanh$ держит числа памяти в интервале $(-1, 1)$. Если нужен ответ на каждом шаге, его считают из памяти обычным линейным слоем: $y_t = V h_t + c$.

*Сквозной пример.* $H = D = 1$, $W = 0{,}5$, $U = 1$, $b = 0$, вход $(1, 0, 0, 0, 0)$:

| $t$ | $x_t$ | $0{,}5\,h_{t-1} + x_t$ | $h_t$ |
|---|---|---|---|
| 1 | 1 | $0 + 1 = 1$ | $\tanh 1 \approx 0{,}7616$ |
| 2 | 0 | $0{,}3808$ | $0{,}3634$ |
| 3 | 0 | $0{,}1817$ | $0{,}1797$ |
| 4 | 0 | $0{,}0899$ | $0{,}0896$ |
| 5 | 0 | $0{,}0448$ | $0{,}0448$ |

Память об импульсе тает примерно вдвое за шаг – это множитель $W = 0{,}5$.

**3. Развертка во времени и общие веса.** Если нарисовать все шаги подряд, получится обычная глубокая сеть из $T$ одинаковых слоев – **развернутый граф вычислений** (Гудфеллоу, раздел 10.1). Главное: на **каждом** шаге используются **одни и те же** $W$, $U$, $b$. Это разделение параметров во времени – так же, как свертка из части 3К делит одно ядро между всеми позициями картинки. Поэтому число параметров не зависит от длины последовательности: $H \cdot H + H \cdot D + H$, и сеть, обученная на фразах из 10 слов, читает и фразы из 50.

**4. Схемы задач.** Из одной ячейки собирают разные схемы:
- **много-к-одному** – прочитать всю последовательность и дать один ответ по последнему состоянию $h_T$ (тональность отзыва; задание Р4);
- **много-ко-многим** – ответ на каждом шаге (следующая буква текста, следующее значение ряда; задания Р5 и Р6);
- **кодировщик-декодер** – одна РНС читает вход и сжимает его в состояние, другая по этому состоянию пишет выход другой длины (машинный перевод; раздел 10.4);
- **двунаправленная** РНС читает последовательность в обе стороны, когда весь вход известен заранее (раздел 10.3).

**5. Обучение: обратное распространение во времени.** Развернутый граф – обычный граф вычислений, и градиенты считает тот же алгоритм из части 2Г, только проход назад идет по шагам времени от конца к началу. Это называют **BPTT** (backpropagation through time). По цепному правилу градиент потери по раннему состоянию – произведение множителей всех шагов:

$$\frac{\partial h_T}{\partial h_0} = \prod_{t=1}^{T}\frac{\partial h_t}{\partial h_{t-1}}, \qquad \frac{\partial h_t}{\partial h_{t-1}} = \big(1 - h_t^2\big)\cdot W \quad \text{(в одномерном случае)}.$$

Так как веса общие, градиенты по $W$ со всех шагов **складываются** – правило ветвления из части 2Г.

**6. Затухание и взрыв градиента.** В одномерном примере каждый множитель не больше $|W| = 0{,}5$ (множитель $1 - h_t^2 \le 1$ еще уменьшает его). Через 10 шагов остается не больше $0{,}5^{10} \approx 0{,}001$: сигнал ошибки почти не доходит до начала, и сеть не может выучить связь между началом и концом последовательности. Это **затухание градиента** (раздел 10.7). Если же $|W| > 1$ и насыщения нет, множители растут: $1{,}5^{10} \approx 57{,}7$ – **взрыв градиента**, обучение идет рывками. В многомерной сети роль $|W|$ играют «коэффициенты растяжения» матрицы $W$. От взрыва помогает **отсечение градиента** (раздел 10.11.1): если норма градиента $\lVert g \rVert$ больше порога $c$, градиент уменьшают, сохраняя направление: $g \leftarrow g \cdot c / \lVert g \rVert$. В PyTorch это `torch.nn.utils.clip_grad_norm_`. От затухания отсечение не спасает – нужна другая ячейка.

**7. LSTM: память с вентилями.** В ячейке LSTM (long short-term memory, раздел 10.10.1) кроме $h_t$ есть отдельная **ячейка памяти** $c_t$ и три **вентиля** – числа от 0 до 1 (сигмоиды), которые решают, что делать с памятью:

$$f_t = \sigma(W_f h_{t-1} + U_f x_t + b_f) \quad \text{– вентиль забывания: сколько старой памяти оставить},$$
$$i_t = \sigma(W_i h_{t-1} + U_i x_t + b_i) \quad \text{– вентиль входа: сколько нового записать},$$
$$g_t = \tanh(W_g h_{t-1} + U_g x_t + b_g) \quad \text{– что именно записать},$$
$$o_t = \sigma(W_o h_{t-1} + U_o x_t + b_o) \quad \text{– вентиль выхода: что сообщить наружу},$$
$$c_t = f_t \odot c_{t-1} + i_t \odot g_t, \qquad h_t = o_t \odot \tanh c_t,$$

где $\odot$ – поэлементное умножение. Главное – формула для $c_t$: память не пересчитывается заново, а **обновляется добавкой**, как в остаточном блоке ResNet из части 4А. Градиент по памяти через шаг равен $\partial c_t / \partial c_{t-1} = f_t$: если вентиль забывания близок к 1, градиент проходит через десятки шагов почти без потерь. Поэтому вентилю забывания часто задают начальное смещение побольше (1–3), чтобы сеть с самого начала «по умолчанию помнила» (раздел 10.10.1; этот прием используется в задании Р4).

**8. GRU – упрощенная LSTM.** Ячейка GRU (раздел 10.10.2) обходится без отдельной памяти $c_t$ и двумя вентилями: вентиль обновления $z_t$ смешивает старое состояние с кандидатом, $h_t = (1 - z_t)\odot h_{t-1} + z_t \odot \tilde h_t$, а вентиль сброса $r_t$ решает, сколько старого состояния использовать при расчете кандидата $\tilde h_t$. Параметров примерно на четверть меньше, чем у LSTM, качество обычно близкое.

**9. Рекуррентные слои в PyTorch.** `nn.RNN(D, H, batch_first=True)` принимает пакет формы $(N, T, D)$ – объекты, шаги, признаки – и возвращает пару: все состояния `out` формы $(N, T, H)$ и последнее состояние `h_n` формы $(1, N, H)$ (первая ось – номер слоя). `nn.LSTM` возвращает `out, (h_n, c_n)`. Веса называются `weight_ih_l0` (это $U$, форма $H \times D$), `weight_hh_l0` (это $W$, $H \times H$), и смещений два – `bias_ih_l0` и `bias_hh_l0` (в формуле они просто складываются). У LSTM веса четырех блоков $i, f, g, o$ хранятся одной матрицей из $4H$ строк в этом порядке.

In [ ]:
# Нужна, только если ноутбук запускается отдельно от ПР
import math
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')   # графики сохраняются в .png и показываются через display; для интерактива закомментируйте
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from IPython.display import Image, display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
print('torch', torch.__version__)

def show(t, k=3):
    """Печать тензора списком с округлением – так удобно сверять с «ожидается»."""
    t = torch.as_tensor(t).detach()
    if t.dim() == 0:
        return round(t.item(), k)
    return torch.round(t.double() * 10**k).div(10**k).tolist()

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
# Внешние данные не нужны: последовательности генерируются, текст для Р6 записан в ноутбуке.

In [ ]:
# Учебный пример со слайдов: h_t = tanh(0.5 * h_{t-1} + 1 * x_t)
W_ex = torch.tensor([[0.5]])      # память -> память, форма (H, H)
U_ex = torch.tensor([[1.0]])      # вход -> память, форма (H, D)
b_ex = torch.tensor([0.0])
x_ex = torch.tensor([1., 0., 0., 0., 0.]).view(1, 5, 1)   # (N = 1 объект, T = 5 шагов, D = 1 признак)
h0_ex = torch.zeros(1, 1)                                  # (N, H)

**Задание Р1. Шаг и проход рекуррентной сети.**

- `rnn_step(x_t, h_prev, W, U, b)` – один шаг по формуле из п. 2 теории для **пакета**: `x_t` имеет форму $(N, D)$, `h_prev` – $(N, H)$. Объекты хранятся строками, поэтому матрицы умножаются справа в транспонированном виде: $h_{t-1}W^\top + x_t U^\top + b$ – так же, как слой в части 1С и как считает `nn.RNN`.
- `rnn_forward(xs, h0, W, U, b)` – проход по всей последовательности `xs` формы $(N, T, D)$: цикл по шагам $t$, на каждом новый `h` из своего `rnn_step` с входом `xs[:, t]` (срез – все объекты, шаг $t$, все признаки). Все состояния собрать в список и склеить по оси времени `torch.stack(hs, dim=1)` – результат $(N, T, H)$.

*Ожидаемые числа* – таблица из п. 2 теории. Вторая проверка – постоянный вход $(1, 1, 1, 1, 1)$: память насыщается и перестает меняться, подходя к числу $h^*$, для которого $h^* = \tanh(0{,}5\,h^* + 1)$, то есть $h^* \approx 0{,}8952$.

In [ ]:
def rnn_step(x_t, h_prev, W, U, b):
    return ...                            # <-- tanh(h_prev W^T + x_t U^T + b)

def rnn_forward(xs, h0, W, U, b):
    h, hs = h0, []
    for t in range(xs.shape[1]):
        h = ...                           # <-- шаг с входом xs[:, t]
        hs.append(h)
    return ...                            # <-- склеить состояния по оси 1


H_ex = rnn_forward(x_ex, h0_ex, W_ex, U_ex, b_ex)
print(tuple(H_ex.shape), '  ожидается (1, 5, 1)')
print(show(H_ex[0, :, 0], 4), '  ожидается [0.7616, 0.3634, 0.1797, 0.0896, 0.0448]')
print(show(rnn_forward(torch.ones(1, 5, 1), h0_ex, W_ex, U_ex, b_ex)[0, :, 0], 4),
      '  ожидается [0.7616, 0.8811, 0.8938, 0.8951, 0.8952]')

**Вопрос Р1.** а) Во сколько раз уменьшается память на каждом шаге без входа и почему именно во столько? б) Сколько параметров у ячейки с памятью $H = 32$ и входом $D = 4$? Изменится ли это число, если последовательность станет в 10 раз длиннее? в) Чем РНС похожа на сверточный слой из части 3К?

*Ваш ответ:*

**Задание Р2. Сверка с `nn.RNN`.**

Функция `copy_to_rnn(rnn, W, U, b)` записывает наши веса в модуль `nn.RNN` (п. 9 теории): `weight_hh_l0` $\leftarrow W$, `weight_ih_l0` $\leftarrow U$, `bias_ih_l0` $\leftarrow b$, а второе смещение `bias_hh_l0` обнуляется (в нашей формуле смещение одно). Копирование – внутри `torch.no_grad()`, как в части 1С; в отличие от `nn.Linear`, транспонировать не нужно: формы у PyTorch те же, что у нас ($H \times H$ и $H \times D$).

Проверяем на примере со слайда и на случайной сети: 2 последовательности по 7 шагов, вход $D = 3$, память $H = 4$. `nn.RNN` принимает начальное состояние формы $(1, N, H)$ – с лишней осью номера слоя, ее добавляет `h0.unsqueeze(0)`.

In [ ]:
def copy_to_rnn(rnn, W, U, b):
    with torch.no_grad():
        rnn.weight_hh_l0.copy_(...)       # <-- W
        rnn.weight_ih_l0.copy_(...)       # <-- U
        rnn.bias_ih_l0.copy_(...)         # <-- b
        rnn.bias_hh_l0.zero_()
    return rnn


rnn1 = copy_to_rnn(nn.RNN(1, 1, batch_first=True), W_ex, U_ex, b_ex)
out1, h_n1 = rnn1(x_ex, h0_ex.unsqueeze(0))
print(show(out1[0, :, 0], 4), '  ожидается [0.7616, 0.3634, 0.1797, 0.0896, 0.0448]')

g = torch.Generator().manual_seed(RANDOM_STATE)
W_r, U_r, b_r = torch.randn(4, 4, generator=g) * 0.5, torch.randn(4, 3, generator=g), torch.randn(4, generator=g)
xs_r, h0_r = torch.randn(2, 7, 3, generator=g), torch.zeros(2, 4)
rnn4 = copy_to_rnn(nn.RNN(3, 4, batch_first=True), W_r, U_r, b_r)
out4, h_n4 = rnn4(xs_r, h0_r.unsqueeze(0))
mine = rnn_forward(xs_r, h0_r, W_r, U_r, b_r)
print(tuple(out4.shape), tuple(h_n4.shape), '  ожидается (2, 7, 4) (1, 2, 4)')
print(f'{(mine - out4).abs().max().item():.1e}', '  ожидается < 1e-06')
print(torch.allclose(h_n4[0], out4[:, -1]), '  ожидается True (h_n – последний столбец out)')

**Вопрос Р2.** а) Зачем у `nn.RNN` два смещения, если в формуле хватает одного? б) Что хранит `h_n` и чем он отличается от `out`? Какой из них нужен для схемы «много-к-одному»?

*Ваш ответ:*

**Задание Р3. Как затухает градиент.**

Функция `grad_h0(xs, h0, W, U, b)` возвращает градиент последнего состояния по начальному: $\partial h_T / \partial h_0$ (п. 5 теории). Порядок как в части 2Г:
1. сделать копию начального состояния, за которой следит autograd: `h0 = h0.clone().requires_grad_(True)`;
2. прогнать свою `rnn_forward`;
3. взять последнее состояние `H[:, -1]`, сложить (`.sum()` – `backward` вызывается у скаляра) и вызвать `.backward()`;
4. вернуть `h0.grad`.

*Три проверки на одномерной сети:*
- вход – импульс из примера. Множители $0{,}5(1 - h_t^2)$ по пяти шагам: $0{,}5 \cdot 0{,}42 \cdot \ldots$, произведение $\approx 0{,}0109$ – меньше, чем $0{,}5^5 \approx 0{,}031$, потому что насыщение $\tanh$ тоже гасит градиент;
- нулевой вход длины 10: состояние все время равно 0, множители ровно $0{,}5$, градиент $0{,}5^{10} \approx 0{,}001$ – **затухание**;
- то же при $W = 1{,}5$: $1{,}5^{10} \approx 57{,}665$ – **взрыв**.

Готовая ячейка рисует $|\partial h_T / \partial h_0|$ в зависимости от длины $T$ для нескольких $W$ в логарифмическом масштабе: при $|W| < 1$ – прямая вниз, при $|W| > 1$ – вверх.

In [ ]:
def grad_h0(xs, h0, W, U, b):
    h0 = ...                              # <-- копия h0 с requires_grad
    H = ...                               # <-- своя rnn_forward
    ...                                   # <-- сумма последнего состояния и backward
    return h0.grad


print(show(grad_h0(x_ex, h0_ex, W_ex, U_ex, b_ex), 4), '  ожидается [[0.0109]]')
zeros10 = torch.zeros(1, 10, 1)
print(show(grad_h0(zeros10, h0_ex, W_ex, U_ex, b_ex), 4), '  ожидается [[0.001]] (0.5^10)')
print(show(grad_h0(zeros10, h0_ex, torch.tensor([[1.5]]), U_ex, b_ex), 3), '  ожидается [[57.665]] (1.5^10)')

In [ ]:
# Готовая ячейка: как градиент зависит от длины последовательности
lengths = list(range(1, 51))
fig, ax = plt.subplots(figsize=(7, 3.4))
for w, c in [(0.5, '#6E6E78'), (0.9, '#9A9AA3'), (1.0, '#1C1C22'), (1.1, '#8F8260')]:
    vals = [abs(grad_h0(torch.zeros(1, T, 1), h0_ex, torch.tensor([[w]]), U_ex, b_ex).item()) for T in lengths]
    ax.plot(lengths, vals, color=c, lw=2, label=f'W = {w}')
ax.set_yscale('log'); ax.set_xlabel('длина последовательности T'); ax.set_ylabel('|dh_T / dh_0|')
ax.legend(); ax.set_title('Затухание и взрыв градиента в РНС')
fig.tight_layout(); fig.savefig('rnn_gradient.png', dpi=120); plt.close(fig)
display(Image('rnn_gradient.png'))

**Вопрос Р3.** а) Почему с импульсом градиент меньше, чем $0{,}5^5$? б) Что будет с обучением, если связь между входом и ответом растянута на 50 шагов, а $|W| = 0{,}9$? Посчитайте множитель. в) Почему отсечение градиента помогает от взрыва, но не от затухания?

*Ваш ответ:*

**Задание Р4. Запомнить первый символ: РНС против LSTM.**

Задача «много-к-одному» (п. 4 теории). Последовательность длины $T$: на первом шаге – один из 4 символов в виде вектора из нулей с единицей на месте номера символа (one-hot), на всех следующих шагах – нули. Ответ – номер первого символа. Угадывание дает точность $0{,}25$. Чтобы ответить, сеть должна пронести информацию через все $T$ шагов.

Класс `SeqClassifier(cell, n_in, hidden, n_out)`:
- в `__init__`: `self.rnn` – `nn.RNN(n_in, hidden, batch_first=True)`, если `cell == 'rnn'`, иначе `nn.LSTM` с теми же аргументами; `self.head` – `nn.Linear(hidden, n_out)`. Для LSTM готовая строка прибавляет 3 к смещению вентиля забывания (строки с $H$ по $2H - 1$ матрицы смещений, п. 7 и 9 теории): $\sigma(3) \approx 0{,}95$, и сеть с самого начала почти не стирает память;
- в `forward(x)`: `out, _ = self.rnn(x)` (у LSTM вторым элементом идет пара `(h_n, c_n)`, поэтому ее отбрасываем одинаково), затем `self.head` от последнего шага `out[:, -1]`.

*Число параметров.* РНС: $32 \cdot 4 + 32 \cdot 32 + 2 \cdot 32 = 1\,216$ плюс голова $32 \cdot 4 + 4 = 132$ – всего $1\,348$. LSTM – вчетверо больше в рекуррентной части: $4 \cdot 1\,216 + 132 = 4\,996$.

Готовая ячейка обучает обе сети на длинах 5, 20 и 50 (Adam, 800 шагов по 64 последовательности, отсечение градиента 1.0) и печатает таблицу точности на 1000 новых последовательностях.

In [ ]:
def make_batch(n, T, n_symbols=4, generator=None):
    """Первый шаг – случайный символ (one-hot), дальше нули. Ответ – номер первого символа."""
    y = torch.randint(0, n_symbols, (n,), generator=generator)
    x = torch.zeros(n, T, n_symbols)
    x[:, 0] = F.one_hot(y, n_symbols).float()
    return x, y

class SeqClassifier(nn.Module):
    def __init__(self, cell='rnn', n_in=4, hidden=32, n_out=4):
        super().__init__()
        if cell == 'rnn':
            self.rnn = ...                          # <-- nn.RNN(n_in, hidden, batch_first=True)
        else:
            self.rnn = ...                          # <-- nn.LSTM с теми же аргументами
            with torch.no_grad():
                self.rnn.bias_ih_l0[hidden:2 * hidden] += 3.0   # вентиль забывания: «по умолчанию помнить»
        self.head = ...                             # <-- линейный слой hidden -> n_out

    def forward(self, x):
        out, _ = ...                                # <-- рекуррентный слой
        return ...                                  # <-- голова от последнего шага


xb, yb = make_batch(8, 5, generator=torch.Generator().manual_seed(0))
print(tuple(xb.shape), show(xb[0].sum()), '  ожидается (8, 5, 4) 1.0')
for cell in ['rnn', 'lstm']:
    m = SeqClassifier(cell)
    print(cell, tuple(m(xb).shape), count_params(m), '  ожидается (8, 4) и 1348 / 4996')

In [ ]:
# Готовая ячейка обучения (около минуты)
def train_seq(cell, T, steps=800, seed=RANDOM_STATE):
    torch.manual_seed(seed)
    model = SeqClassifier(cell)
    opt = torch.optim.Adam(model.parameters(), lr=0.003)
    gen = torch.Generator().manual_seed(seed)
    for step in range(steps):
        x, y = make_batch(64, T, generator=gen)
        opt.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    x, y = make_batch(1000, T, generator=torch.Generator().manual_seed(seed + 1))
    with torch.no_grad():
        return (model(x).argmax(dim=1) == y).float().mean().item()

acc = {}
start = time.perf_counter()
for T in [5, 20, 50]:
    for cell in ['rnn', 'lstm']:
        acc[(cell, T)] = train_seq(cell, T)
print(f'{"T":>4}{"RNN":>8}{"LSTM":>8}')
for T in [5, 20, 50]:
    print(f'{T:>4}{acc[("rnn", T)]:>8.3f}{acc[("lstm", T)]:>8.3f}')
print(f'время: {time.perf_counter() - start:.0f} с')
print(acc[('rnn', 5)] >= 0.9, acc[('lstm', 50)] >= 0.9, '  ожидается True True')
print(acc[('rnn', 50)] < 0.9, '  ожидается True (простая РНС на длине 50 обычно близка к угадыванию 0.25)')

**Вопрос Р4.** а) Объясните результат РНС на длине 50 через п. 6 теории. б) Что делает прибавка 3 к смещению вентиля забывания? Проверьте, что будет без нее (закомментируйте строку). в) Почему точность проверяется на новых последовательностях, а не на тех, на которых сеть училась?

*Ваш ответ:*

**Задание Р5. Прогноз зашумленной синусоиды.**

Временной ряд: $s_t = \sin(0{,}2\,t) + 0{,}1\,\varepsilon_t$, где $\varepsilon_t$ – случайный шум, $t = 0, \dots, 999$. По 20 последним значениям нужно предсказать следующее – задача регрессии «много-к-одному».

Функция `make_windows(series, window)` нарезает ряд на пары «окно – следующее значение»: для каждого $i$ от 0 до $n - \text{window} - 1$ окно `series[i : i + window]` и ответ `series[i + window]`. Вернуть `X` формы $(N, \text{window}, 1)$ и `y` формы $(N, 1)$ в `float32`, где $N = n - \text{window}$. Удобно собрать окна списком и склеить `torch.stack`, лишнюю ось признака добавить `.unsqueeze(-1)`.

*Наивный прогноз* «завтра будет как сегодня» (последнее значение окна) дает на тесте среднеквадратичную ошибку около $0{,}037$. Нижний предел для любой модели – дисперсия шума $0{,}1^2 = 0{,}01$: случайную часть предсказать нельзя. Готовая ячейка обучает РНС с памятью 32 и сравнивает.

Данные делятся по времени: первые 760 окон – обучение, остальные – тест. Перемешивать ряд перед разбиением нельзя: модель не должна «подсматривать» в будущее.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
t_axis = np.arange(1000)
series = torch.tensor(np.sin(0.2 * t_axis) + 0.1 * rng.standard_normal(1000), dtype=torch.float32)

def make_windows(series, window):
    xs, ys = [], []
    for i in range(len(series) - window):
        xs.append(...)                    # <-- окно длины window, начиная с i
        ys.append(...)                    # <-- следующее значение после окна
    X = ...                               # <-- склеить окна и добавить ось признака: (N, window, 1)
    y = ...                               # <-- склеить ответы: (N, 1)
    return X, y


X_s, y_s = make_windows(series, 20)
print(tuple(X_s.shape), tuple(y_s.shape), '  ожидается (980, 20, 1) (980, 1)')
print(torch.equal(X_s[1, :, 0], series[1:21]), torch.equal(y_s[0, 0], series[20]), '  ожидается True True')
Xtr_s, ytr_s, Xte_s, yte_s = X_s[:760], y_s[:760], X_s[760:], y_s[760:]
naive_mse = ((Xte_s[:, -1] - yte_s) ** 2).mean().item()
print(round(naive_mse, 3), '  ожидается 0.037')

In [ ]:
# Готовая ячейка обучения прогнозной РНС
class SeqRegressor(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.rnn = nn.RNN(1, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1])

torch.manual_seed(RANDOM_STATE)
reg = SeqRegressor()
opt = torch.optim.Adam(reg.parameters(), lr=0.01)
gen = torch.Generator().manual_seed(RANDOM_STATE)
for step in range(300):
    idx = torch.randint(0, len(Xtr_s), (64,), generator=gen)
    opt.zero_grad()
    loss = F.mse_loss(reg(Xtr_s[idx]), ytr_s[idx])
    loss.backward()
    opt.step()
with torch.no_grad():
    pred = reg(Xte_s)
test_mse = F.mse_loss(pred, yte_s).item()
print(f'РНС: {test_mse:.4f}   наивный прогноз: {naive_mse:.4f}')
print(test_mse < 0.02, test_mse < naive_mse, '  ожидается True True')

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(yte_s[:100, 0], color='#9A9AA3', lw=1.5, label='ряд с шумом')
ax.plot(pred[:100, 0], color='#8F8260', lw=2, label='прогноз РНС')
ax.plot(np.sin(0.2 * t_axis[780:880]), color='#1C1C22', lw=1, ls='--', label='синусоида без шума')
ax.set_xlabel('шаг теста'); ax.legend(); ax.set_title('Прогноз на шаг вперед')
fig.tight_layout(); fig.savefig('sine_forecast.png', dpi=120); plt.close(fig)
display(Image('sine_forecast.png'))

**Вопрос Р5.** а) Почему ошибка РНС не может опуститься ниже $0{,}01$? б) Почему прогноз РНС на графике ближе к синусоиде без шума, чем к зашумленному ряду? в) Сравните с моделями 9 класса: как решить эту задачу линейной регрессией по 20 последним значениям?

*Ваш ответ:*

**Задание Р6. Символьная модель текста.**

Схема «много-ко-многим» (п. 4 теории): сеть читает текст по одной букве и на каждом шаге предсказывает **следующую** букву. Обученную модель можно запустить на генерацию: дать ей затравку, выбрать следующую букву по предсказанным вероятностям, добавить к тексту и повторить (см. [KARPATHY-RNN] в источниках к занятию).

Текст для обучения записан в ноутбуке (около 700 символов). Алфавит – все различные символы текста, отсортированные; словари `stoi` («символ → номер») и `itos` («номер → символ») готовы.
- `encode(text)` – тензор номеров символов (`torch.long`);
- `decode(ids)` – обратно строка (`''.join`);
- `sample_next(logits, temperature, generator)` – номер следующего символа: вероятности $p = \mathrm{softmax}(\text{logits} / \tau)$ (п. 5 теории части 1С), затем случайный выбор по этим вероятностям `torch.multinomial(p, 1, generator=generator)`. **Температура** $\tau$ управляет смелостью: при малой $\tau$ почти всегда берется самая вероятная буква (текст повторяет обучающий), при большой – распределение выравнивается и текст становится случайнее.

Готовые ячейки: модель `CharLSTM` (буква кодируется one-hot вектором, LSTM с памятью 128, линейный слой на размер алфавита), обучение на случайных кусках по 40 символов и генерация по затравке. До обучения потеря близка к $\ln V$, где $V$ – размер алфавита: сеть считает все буквы равновероятными.

In [ ]:
TEXT_R6 = (
    'Беспилотный автомобиль видит дорогу камерами и лидаром. Нейросеть находит на картинке '
    'машины, пешеходов и знаки. Другая сеть следит за тем, как они движутся, и предсказывает, '
    'куда они поедут через секунду. Машина помнит, что было раньше: пешеход, который шагнул к '
    'переходу, скорее всего перейдет дорогу. Поэтому для движения нужна память, а память дают '
    'рекуррентные сети. Они читают данные шаг за шагом и хранят главное в скрытом состоянии. '
    'Простая сеть быстро забывает начало, а сеть с вентилями помнит долго. Вентиль забывания '
    'решает, что стереть, вентиль входа – что записать, вентиль выхода – что сообщить дальше. '
    'Так сеть учится понимать речь, переводить текст и писать новые фразы по одной букве.'
)
chars = sorted(set(TEXT_R6))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(text):
    return ...                            # <-- тензор номеров символов, dtype=torch.long

def decode(ids):
    return ...                            # <-- строка из символов по номерам

def sample_next(logits, temperature=1.0, generator=None):
    p = ...                               # <-- softmax от logits / temperature
    return ...                            # <-- номер из torch.multinomial(p, 1, generator=generator), числом int


data = encode(TEXT_R6)
print(len(TEXT_R6), V, round(math.log(V), 2), '  ожидается 702 43 3.76')
print(decode(data[:11].tolist()), decode(encode('сеть').tolist()) == 'сеть', '  ожидается «Беспилотный» True')
print(sample_next(torch.tensor([0., 10., 0.]), 1.0, torch.Generator().manual_seed(0)), '  ожидается 1')

In [ ]:
# Готовая ячейка: модель и обучение (около 30 секунд)
class CharLSTM(nn.Module):
    def __init__(self, vocab, hidden=128):
        super().__init__()
        self.vocab = vocab
        self.lstm = nn.LSTM(vocab, hidden, batch_first=True)
        self.head = nn.Linear(hidden, vocab)

    def forward(self, ids, state=None):
        out, state = self.lstm(F.one_hot(ids, self.vocab).float(), state)
        return self.head(out), state

torch.manual_seed(RANDOM_STATE)
char_model = CharLSTM(V)
opt = torch.optim.Adam(char_model.parameters(), lr=0.005)
gen = torch.Generator().manual_seed(RANDOM_STATE)
L_SEQ, history = 40, []
for step in range(300):
    starts = torch.randint(0, len(data) - L_SEQ - 1, (32,), generator=gen)
    x = torch.stack([data[s:s + L_SEQ] for s in starts])
    y = torch.stack([data[s + 1:s + L_SEQ + 1] for s in starts])     # та же строка, сдвинутая на букву
    opt.zero_grad()
    logits, _ = char_model(x)
    loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
    loss.backward()
    nn.utils.clip_grad_norm_(char_model.parameters(), 5.0)
    opt.step()
    history.append(loss.item())
print(round(history[0], 1), '  ожидается около 3.8 (ln V)')
print(history[-1] < 1.0, '  ожидается True (потеря ниже 1.0)')

In [ ]:
# Готовая ячейка генерации по затравке
def generate(model, prompt, n_chars=150, temperature=0.5, seed=RANDOM_STATE):
    gen = torch.Generator().manual_seed(seed)
    model.eval()
    ids = encode(prompt).unsqueeze(0)
    with torch.no_grad():
        logits, state = model(ids)                   # прочитать затравку целиком
        out = prompt
        for _ in range(n_chars):
            nxt = sample_next(logits[0, -1], temperature, gen)
            out += itos[nxt]
            logits, state = model(torch.tensor([[nxt]]), state)   # дальше по одной букве, память переносится
    return out

for tau in [0.3, 1.0, 2.0]:
    print(f'температура {tau}:\n{generate(char_model, "Нейросеть ", temperature=tau)}\n')

**Вопрос Р6.** а) Как меняется текст при температуре 0.3, 1.0 и 2.0? Объясните через формулу softmax с делением на $\tau$. б) Почему модель часто повторяет куски обучающего текста? Что нужно, чтобы она писала новые осмысленные фразы? в) Зачем при генерации переносить состояние `state` от шага к шагу?

*Ваш ответ:*

### Задание повышенной сложности (к части 5Р)

1. **Ячейка LSTM на тензорах.** Напишите `lstm_step(x_t, h, c, W_ih, W_hh, b_ih, b_hh)` по формулам из п. 7 теории (матрицы вентилей – срезы по $H$ строк в порядке $i, f, g, o$) и `lstm_forward`. Сверьте с `nn.LSTM`, скопировав его веса; расхождение должно быть меньше $10^{-6}$.
2. **Отсечение градиента.** Обучите простую РНС из Р4 на длине 50 без `clip_grad_norm_` и с ним, записывая норму градиента на каждом шаге (`clip_grad_norm_` возвращает норму до отсечения). Постройте графики нормы и потерь. Бывают ли всплески?
3. **GRU.** Добавьте в `SeqClassifier` вариант `nn.GRU` и заполните строку таблицы Р4. Сколько у него параметров по сравнению с LSTM?
4. **Своя затравка.** Обучите `CharLSTM` на своем тексте в 3–5 тысяч символов (например, на правилах дорожного движения) и сравните генерацию при разной температуре.

In [ ]:
# Место для задания повышенной сложности